In [ ]:
from pathlib import Path
import json

import pandas as pd

from jazz_graph.data import fetch
from jazz_graph.recommendation.playlist import SpotifyListens, ListenField, ListenSchema
from jazz_graph.data.fetch import fetch_recording_traits
from jazz_graph.model.model import UnsupervisedJazzModel
from jazz_graph.data.graph_builder.make_jazz import JazzDataStore, make_jazz_graph_with_style_and_edges


In [ ]:
import torch
#from jazz_graph.recommendation import recommender
from jazz_graph.training.logging import load_model
from jazz_graph.recommendation.recommender import InferenceRecommender


In [ ]:
recording_traits = fetch_recording_traits().set_index('recording_id')
graph_data = make_jazz_graph_with_style_and_edges(JazzDataStore('/workspace/local_data/graph_parquet'))

In [ ]:
spotify = SpotifyListens(recording_traits, ListenSchema([
    ListenField('artist', 'artistName'), ListenField('title', 'trackName')
]))
apple = SpotifyListens(recording_traits, ListenSchema([
    ListenField('artist', 'Artist'), ListenField('title', 'Title'), ListenField('album', 'Album')
]))

In [ ]:
with open('/workspace/local_data/spotify_dataset/friends.json', 'r') as f:
    friends = json.load(f)
spotify_paths = friends['spotify']
apple_paths = friends['apple']

In [ ]:
class RecommendationMaker:
    def __init__(self, recommender: InferenceRecommender, recording_traits: pd.DataFrame):
        self.recommender = recommender
        self.recording_traits = recording_traits

    def make_recommendations(self, listens: SpotifyListens, data: list[dict], n_recommendations: int = 30, directory: str|Path|None = None):
        ids = listens.get_listen_ids(data)
        rec_basis = self.recording_traits.loc[ids]
        recs, _, mask = self.recommender.get_recommendations(ids)
        recs = recs[~mask][:n_recommendations]
        recommendations = recording_traits.loc[recs]
        if directory:
            self.write_recommendations(directory, recommendations, rec_basis)
            print(f"Recommendations written to {directory}")
        else:
            return recommendations, rec_basis

    def write_recommendations(self, directory, recommendations, rec_basis):
        directory = Path(directory)
        recommendations.to_csv(directory / 'recommendations.csv', index=False)
        rec_basis.to_csv(directory / 'known_jazz.csv', index=False)


In [ ]:

class UnsupervisedModelAdapter(torch.nn.Module):
    def __init__(self, model: UnsupervisedJazzModel):
        super().__init__()
        self.model = model

    def __call__(self, x_dict, edge_index_dict, batch):
        return self.model(batch)
        return self.model.encode(batch)


def get_model(model_path):
    with open(Path(model_path) / 'config.json', 'r') as f:
        run_config = json.loads(f.read())
    model_state = load_model(model_path)
    model_state = model_state.get('model_state_dict', model_state)
    model = UnsupervisedJazzModel.from_config(run_config)
    model.load_state_dict(model_state)
    model = UnsupervisedModelAdapter(model)
    return model

model_path = '/workspace/experiments/2026-04-03_16-48-18_gnn_simCLR_graph_parquet'
recommender = InferenceRecommender(get_model(model_path), graph_data, pooling='softmax')
rec_maker = RecommendationMaker(recommender, recording_traits)

In [ ]:
def extract_spotify_history(path: str | Path):
    path = Path(path)
    histories = [p for p in path.iterdir() if p.name.startswith('StreamingHistory_m')]

    spotify_data = []
    for history in histories:
        with open(history, 'r') as f:
            data = json.load(f)
        spotify_data.extend(data)
    return spotify_data


In [ ]:
for path in spotify_paths:
    data = extract_spotify_history(path)
    directory = Path(path).parent
    rec_maker.make_recommendations(spotify, data, directory=directory)


### Apple

In [ ]:

for path in apple_paths:
    with open(path, 'r') as f:
        data = json.load(f)
    directory = '/'.join(path.split('/')[:5])
    rec_maker.make_recommendations(apple, data, directory=directory)


In [ ]:
friends = {'spotify': spotify_paths, 'apple': apple_paths}
with open('/workspace/local_data/spotify_dataset/friends.json', 'w') as f:
    json.dump(friends, f)


## Collective Recommendations

In [ ]:
import numpy as np
listens = np.array([], dtype=np.int64)
for path in spotify_paths:
    data = extract_spotify_history(path)
    ids = spotify.get_listen_ids(data)
    listens = np.union1d(ids, listens)
for path in apple_paths:
    with open(path, 'r') as f:
        data = json.load(f)
    ids = apple.get_listen_ids(data)
    listens = np.union1d(ids, listens)

In [ ]:
f"As a group, there are {listens.size} unique jazz performances we listened to."


In [ ]:

# recs, _, mask = recommender.get_recommendations(listens)
recording_traits.loc[recs[~mask]][:15]#.album.unique()
